## Finetuning for Instruction QnA

In [29]:
import torch
import torch.nn as nn

In [ ]:
import tiktoken
tokenizer = tiktoken.get_encoding("gpt2")

In [1]:
import json
with open('instruction-data.json', 'r') as f:
    data = json.load(f)

In [4]:
count = 0
for element in data:
    if element['input']:
        count += 1
count, len(data)

(325, 1100)

In [9]:
def format_input(entry):
    instruction_text = (
    f"Below is an instruction that describes a task. "
    f"Write a response that appropriately completes the request."
    f"\n\n### Instruction:\n{entry['instruction']}"
    )
    input_text = (
    f"\n\n### Input:\n{entry['input']}" if entry["input"] else ""
    )
    return instruction_text + input_text


In [10]:
model_input = format_input(data[50])
desired_response = f"\n\n### Response:\n{data[50]['output']}"
print(model_input + desired_response)

Below is an instruction that describes a task. Write a response that appropriately completes the request.

### Instruction:
Identify the correct spelling of the following word.

### Input:
Ocassion

### Response:
The correct spelling is 'Occasion.'


In [11]:
train_data = data[:int(len(data)*0.85)]
val_data = data[int(len(data)*0.85): int(len(data)*0.95)]
test_data = data[int(len(data)*0.95):]


In [12]:
len(train_data), len(test_data), len(val_data)

(935, 55, 110)

### Creating Batches and DataLoaders

In [28]:
class InstructDataset:
    def __init__(self, data, tokenizer):
        self.data = data
        self.encoded_texts = []
        for entry in data:
            instruction_plus_input = format_input(entry)
            response_text = f"\n\n### Response:\n{entry['output']}"
            self.encoded_texts.append(tokenizer.encode(instruction_plus_input + response_text))

    def __getitem__(self, index):
        return self.encoded_texts[index]
    def __len__(self):
        return len(self.data)



In [20]:
train_dataset = InstructDataset(
    train_data
)

In [36]:
def custom_collate_fn_1(batch, pad_token_id = 50256, device = "cpu"):
    batch_max_length = max((len(item)) for item in batch)
    inputs_list = []
    for item in batch:
        if len(item) == batch_max_length:
            pass
        else:
            item += [pad_token_id]*(batch_max_length - len(item))
        inputs = torch.tensor(item)
        inputs_list.append(inputs)
    inputs_tensor = torch.stack(inputs_list).to(device)
    return inputs_tensor

    


In [37]:
inputs_1 = [0, 1, 2, 3, 4]
inputs_2 = [5, 6]
inputs_3 = [7, 8, 9]
inputs_4 = []
batch = (
inputs_1,
inputs_2,
inputs_3,
inputs_4
)
print(custom_collate_fn_1(batch))

tensor([[    0,     1,     2,     3,     4],
        [    5,     6, 50256, 50256, 50256],
        [    7,     8,     9, 50256, 50256],
        [50256, 50256, 50256, 50256, 50256]])


In [50]:
from copy import copy
def custom_collate_fn_3(batch, pad_token_id = 50256, device = "cpu"):
    batch_max_length = max((len(item)) for item in batch)
    inputs_list, targets_list = [],[]
    for item in batch:
        item_copy = item.copy()
        if len(item) == batch_max_length:
            pass
        else:
            item += [pad_token_id]*(batch_max_length - len(item))
        inputs = torch.tensor(item)
        #targets = torch.tensor(item[1:] + [pad_token_id])
        targets = torch.tensor(item_copy[1:] + [pad_token_id] + [-100]*(batch_max_length - len(item_copy)))
        print(targets)
        inputs_list.append(inputs)
        targets_list.append(targets)

    inputs_tensor = torch.stack(inputs_list).to(device)
    targets_tensor = torch.stack(targets_list).to(device)
    return inputs_tensor, targets_tensor

In [52]:
inputs_1 = [0, 1, 2, 3, 4]
inputs_2 = [5, 6]
inputs_3 = [7, 8, 9]
batch = (
inputs_1,
inputs_2,
inputs_3
)
x,y = custom_collate_fn_3(batch)

tensor([    1,     2,     3,     4, 50256])
tensor([    6, 50256,  -100,  -100,  -100])
tensor([    8,     9, 50256,  -100,  -100])


In [53]:
x

tensor([[    0,     1,     2,     3,     4],
        [    5,     6, 50256, 50256, 50256],
        [    7,     8,     9, 50256, 50256]])

In [54]:
y

tensor([[    1,     2,     3,     4, 50256],
        [    6, 50256,  -100,  -100,  -100],
        [    8,     9, 50256,  -100,  -100]])